# Fundamento 1 — De dónde viene el dataset

## Explicación

Este notebook es la versión hands-on de la guía [`01-pipeline-dataset.md`](01-pipeline-dataset.md). Reproduce lo que haría un Data Scientist **antes** de escribir una sola línea de código de Machine Learning: cargar el dataset, entender su tamaño, revisar si contiene información personal identificable (PII), simular el pipeline ETL (Extract, Transform, Load) que lo produjo, y revisar el balance de clases del target.

## Por qué es importante

Un modelo entrenado sobre datos mal entendidos (sucios, sesgados, o con problemas de privacidad) es un riesgo, no una solución. Antes de tocar `scikit-learn` hay que confirmar: cuántas filas/columnas hay, qué tipo de dato tiene cada una, si el target está balanceado, y si el dataset respeta la privacidad de las personas.

## Mapa del notebook (para no perderse)

Aquí todavía **no hay modelo ni predicciones**. Solo hacemos preguntas sobre el dataset, en este orden:

| Sección | La pregunta que responde | Qué obtienes |
|---|---|---|
| 0. Preparativos | ¿Están las librerías y existe el CSV? | Un `assert` que falla rápido si falta el archivo |
| 1. Cargar y conocer | ¿Cómo son los datos? | `df` cargado, primeras filas y tipos de dato |
| 2. Tamaño | ¿Cuánto pesa en disco y en RAM? | Dos números en MB |
| 3. Identificadores y PII | ¿Hay datos personales? | Lista de columnas + verificación del ID |
| 4. Simular ETL | ¿Cómo se produce el CSV que usa el equipo de ML? | `raw_ingested.csv` en `datasets/processed/` |
| 5. Balance de clases | ¿Qué proporción de empleados se fue? | Porcentaje "Stayed" vs "Left" |

## Dos ideas que conviene tener claras antes de empezar

- **Este dataset ya llegó "cocinado"**: en una empresa real vendría de un ETL sobre HRMS, nómina, LMS y evaluaciones. Aquí simulamos ese proceso en pequeño para entenderlo, no para construirlo.
- **`Attrition` es el target**: la columna que más adelante querremos predecir. Todo lo demás son candidatos a features.

## Librerías que usamos

| Librería | Para qué la usamos aquí |
|---|---|
| `os` (estándar de Python) | Consultar el tamaño del archivo CSV en disco (`os.path.getsize`) |
| `pathlib.Path` (estándar de Python) | Construir rutas de archivo de forma segura y multiplataforma |
| `pandas` | Cargar el CSV como `DataFrame`, inspeccionarlo y guardar resultados |

## Comandos / funciones clave de este notebook

| Comando | Qué hace |
|---|---|
| `pd.read_csv(...)` | Carga un archivo CSV en un `DataFrame` de pandas |
| `df.head()` | Muestra las primeras 5 filas del DataFrame |
| `df.info()` | Resume tipo de dato, cantidad de nulos y memoria usada por columna |
| `df.shape` | Tupla `(filas, columnas)` del DataFrame |
| `os.path.getsize(...)` | Tamaño de un archivo en bytes |
| `df.memory_usage(deep=True)` | Memoria real (bytes) que ocupa cada columna en RAM |
| `df["col"].is_unique` | `True` si no hay valores repetidos en esa columna |
| `df.drop(columns=[...])` | Elimina una o varias columnas y devuelve un nuevo DataFrame |
| `df.to_csv(...)` | Guarda un DataFrame como archivo CSV |
| `df["col"].value_counts(normalize=True)` | Cuenta ocurrencias de cada valor único; `normalize=True` las expresa como proporción (0-1) |

Guía de referencia: [`01-pipeline-dataset.md`](01-pipeline-dataset.md)


## 0. Preparativos: librerías y ruta del dataset

Esta celda no analiza nada todavía, solo prepara el terreno:

1. **Importa** `os` (tamaño de archivos), `pathlib.Path` (rutas que funcionan igual en Windows y Linux) y `pandas` (la librería de tablas).
2. **`pd.set_option("display.max_columns", None)`**: por defecto pandas oculta columnas cuando la tabla es ancha; esto le dice que las muestre todas.
3. **Define `RAW_DATA_PATH`**, la ruta al CSV del proyecto, escrita *relativa a esta carpeta*. Por eso el notebook debe abrirse desde el repositorio clonado.
4. **`assert ... .exists()`**: si el archivo no está, la celda falla aquí con un mensaje claro, en vez de fallar tres celdas más abajo con un error confuso. Es el patrón "fail fast".

**Qué mirar en la salida:** nada. Si no aparece ningún error, todo correcto. Si salta el `AssertionError`, revisa desde dónde abriste VS Code.


In [8]:
import os  # librería estándar: nos deja consultar el sistema de archivos (tamaño de un archivo, rutas del SO, etc.)
from pathlib import Path  # forma moderna y multiplataforma (Windows/Linux/Mac) de construir rutas de archivo
import pandas as pd  # librería principal para cargar y manipular datos tabulares (DataFrames)

pd.set_option("display.max_columns", None)  # evita que pandas oculte columnas al imprimir un DataFrame ancho

# Ruta al CSV crudo del proyecto, relativa a esta carpeta (01-fundamentosml-datascience/)
RAW_DATA_PATH = Path("../02-phase-1-local-dev-mlops/datasets/employee_attrition.csv")
# Falla rápido y con un mensaje claro si el dataset no existe en la ruta esperada
assert RAW_DATA_PATH.exists(), f"No se encontró el dataset en {RAW_DATA_PATH.resolve()}"

## 1. Cargar y conocer el dataset

Igual que hace `01_ingestion.py`: cargar el CSV y confirmar que se leyó bien **antes** de tocar nada.

`pd.read_csv()` lee el archivo entero y lo convierte en un **DataFrame**: una tabla en memoria, con filas y columnas, parecida a una hoja de cálculo pero manipulable desde código.

`df.head()` muestra las primeras 5 filas. No es un adorno: es la comprobación de que las columnas se separaron bien, que los acentos se leyeron correctamente y que los valores tienen sentido.

**Qué mirar en la salida:** que aparezcan columnas con nombre propio (`Employee ID`, `Age`, `Attrition`…) y no una sola columna gigante con comas dentro (eso significaría un separador mal detectado).


In [10]:
df = pd.read_csv(RAW_DATA_PATH)  # lee el CSV completo en memoria como un DataFrame de pandas
df.head()  # muestra las primeras 5 filas para una primera inspección visual

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Overtime,Distance from Home,Education Level,Marital Status,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition,dataset_type
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,No,22,Associate Degree,Married,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed,train
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,No,21,Master’s Degree,Divorced,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed,train
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,No,11,Bachelor’s Degree,Married,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed,train
3,65791,36,Female,7,Education,3989,Good,High,High,1,No,27,High School,Single,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed,train
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,Yes,71,High School,Divorced,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed,train


### La radiografía del dataset: `df.info()`

Si `head()` es la foto, `info()` es la radiografía. Devuelve, para cada columna: su **nombre**, cuántos valores **no nulos** tiene y su **tipo de dato**.

**Cómo leer los tipos:**

| Lo que ves | Qué significa |
|---|---|
| `int64` / `float64` | Columna numérica: el modelo la puede usar tal cual |
| `object` | Normalmente texto: **habrá que convertirla a número** en el Fundamento 2 |
| `non-null count` menor al total de filas | Esa columna tiene valores faltantes |

**Qué mirar en la salida:** cuántas columnas son `object` (esas son el trabajo del feature engineering) y si algún `non-null count` no coincide con el número total de filas (eso sería trabajo de la etapa de limpieza). Al final aparece también la memoria total usada, que enlaza con la siguiente sección.


In [11]:
df.info()  # resumen técnico: tipo de dato de cada columna, cuántos valores no nulos tiene y memoria total usada

<class 'pandas.DataFrame'>
RangeIndex: 74498 entries, 0 to 74497
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Employee ID               74498 non-null  int64
 1   Age                       74498 non-null  int64
 2   Gender                    74498 non-null  str  
 3   Years at Company          74498 non-null  int64
 4   Job Role                  74498 non-null  str  
 5   Monthly Income            74498 non-null  int64
 6   Work-Life Balance         74498 non-null  str  
 7   Job Satisfaction          74498 non-null  str  
 8   Performance Rating        74498 non-null  str  
 9   Number of Promotions      74498 non-null  int64
 10  Overtime                  74498 non-null  str  
 11  Distance from Home        74498 non-null  int64
 12  Education Level           74498 non-null  str  
 13  Marital Status            74498 non-null  str  
 14  Number of Dependents      74498 non-null  int64
 

## 2. ¿Cuánto "pesa" este dataset?

Referencia del fundamento: de ~0.8 TB de datos crudos a ~300 MB de CSV final. Aquí comparamos dos tamaños distintos que la gente suele confundir:

| Medida | Qué mide | Cómo se obtiene |
|---|---|---|
| **En disco** | Lo que ocupa el archivo `.csv` guardado | `os.path.getsize()` (devuelve bytes) |
| **En memoria** | Lo que ocupa una vez cargado como DataFrame | `df.memory_usage(deep=True).sum()` |

Ambos se dividen entre `1024 ** 2` para pasar de bytes a MB.

**Por qué en memoria pesa más:** en el CSV, el número `35` son dos caracteres de texto; en pandas, es un entero de 64 bits, y cada texto arrastra además la estructura interna del objeto de Python. El parámetro `deep=True` es importante: sin él, pandas no cuenta el contenido real de las cadenas de texto y el número sale engañosamente bajo.

**Por qué esto le importa a MLOps:** es lo que determina cuánta RAM hay que pedirle al pod, al worker de Airflow o a la máquina de entrenamiento. La regla práctica es reservar varias veces el tamaño del archivo en disco.

**Qué mirar en la salida:** los dos números y su diferencia. Con este dataset son pocos MB; imagina la misma proporción con el CSV de 300 MB del caso real.


In [12]:
# os.path.getsize devuelve el tamaño del archivo en bytes; se divide entre 1024^2 para convertir a MB
size_on_disk_mb = os.path.getsize(RAW_DATA_PATH) / (1024 ** 2)
# memory_usage(deep=True) suma el tamaño real en RAM de cada columna, incluyendo el contenido de los strings
size_in_memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Tamaño en disco:   {size_on_disk_mb:,.2f} MB")  # cuánto pesa el archivo .csv en el disco
print(f"Tamaño en memoria: {size_in_memory_mb:,.2f} MB (pandas usa más espacio que el CSV crudo)")  # cuánto ocupa una vez cargado en RAM

Tamaño en disco:   9.52 MB
Tamaño en memoria: 71.27 MB (pandas usa más espacio que el CSV crudo)


## 3. Identificadores y PII (Información Personal Identificable)

Antes de seguir, hay que confirmar que el dataset no trae nombres, correos ni teléfonos, y que los identificadores están pseudonimizados. Esto **no es burocracia**: usar PII para entrenar puede incumplir el GDPR o la normativa local, y el problema aparece cuando el modelo ya está en producción.

Qué hace la celda siguiente:

1. **Imprime la lista de columnas**, para revisarlas una a una buscando datos personales.
2. **Comprueba `df['Employee ID'].is_unique`**: si es `True`, cada fila corresponde a un empleado distinto (no hay duplicados ni filas repetidas por error de un `JOIN` en el ETL).

**Tres conceptos que se suelen mezclar:**

| Término | Qué es | Ejemplo aquí |
|---|---|---|
| **PII** | Identifica directamente a una persona | Nombre, email, teléfono → **no deben estar** |
| **Pseudonimizado** | Sustituye la identidad por un código | `Employee ID` = 1, 2, 3… |
| **Dato sensible** | No identifica, pero es delicado | Salario, datos de salud → se agregan o se enmascaran |

**Qué mirar en la salida:** que ninguna columna sea un nombre/correo/teléfono, y que la comprobación de unicidad diga `True`.

> Ojo con la trampa: `Employee ID` es seguro desde el punto de vista de privacidad, pero **no debe usarse como feature**. Es solo un número correlativo; si el modelo lo usara, aprendería ruido puro. Por eso se elimina en el Fundamento 2.


In [13]:
print("Columnas del dataset:")  # encabezado informativo antes de listar las columnas
print(list(df.columns))  # convierte el índice de columnas de pandas en una lista simple para imprimir

# "Employee ID" es un identificador pseudonimizado (un número secuencial), no un dato personal en sí mismo
print(f"\n¿'Employee ID' es único por fila? {df['Employee ID'].is_unique}")  # is_unique confirma que no hay IDs repetidos
print("No hay columnas de nombre, correo o teléfono: el dataset ya llegó anonimizado, "
      "como se espera después de un pipeline ETL responsable.")  # conclusión manual de la inspección de columnas

Columnas del dataset:
['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition', 'dataset_type']

¿'Employee ID' es único por fila? True
No hay columnas de nombre, correo o teléfono: el dataset ya llegó anonimizado, como se espera después de un pipeline ETL responsable.


## 4. Simulando las 3 etapas de un pipeline ETL

En un caso real, `Extract` conecta con HRMS/nómina/LMS, `Transform` limpia y unifica, y `Load` guarda el resultado. La celda siguiente hace exactamente eso, en miniatura y sobre un solo archivo:

| Etapa | En una empresa real | En esta celda |
|---|---|---|
| **Extract** | Consultas a bases SQL, exports XML/JSON de 4 sistemas | `df.copy()` — el "crudo" ya está cargado |
| **Transform** | Unir fuentes, deduplicar, anonimizar, unificar formatos | Quitar la columna auxiliar `dataset_type` |
| **Load** | Escribir en S3 o en el data lake del equipo de ML | `to_csv()` en `datasets/processed/raw_ingested.csv` |

**Detalles del código que conviene señalar:**

- **`df.copy()` en vez de trabajar sobre `df`**: sin la copia, modificar el "extraído" alteraría también el original. Es la versión pandas de la regla "el dato de origen es inmutable".
- **`mkdir(parents=True, exist_ok=True)`**: crea la carpeta destino junto con las intermedias, y no falla si ya existe, así la celda se puede reejecutar sin errores.
- **`index=False`**: evita que pandas añada una columna extra con el número de fila, que ensuciaría el CSV para la siguiente etapa.

**Qué mirar en la salida:** el número de columnas después del Transform (una menos) y la ruta absoluta del archivo generado. Ese `raw_ingested.csv` es justo el punto de partida del Fundamento 2.

> Idea para llevarse: en producción estas tres líneas serían tres tareas de un DAG de Airflow, con reintentos, alertas y horario de ejecución. El concepto es el mismo; cambia la escala y el arnés que lo rodea.


In [14]:
# Extract: leer el dato "crudo" (ya lo hicimos arriba)
df_extracted = df.copy()  # copiamos el DataFrame para no modificar el original por accidente

# Transform: en un caso real aquí se combinarían fuentes (HRMS, nómina, LMS...).
# Aquí solo quitamos 'dataset_type', una bandera de split que no es una feature real del empleado.
df_transformed = df_extracted.drop(columns=["dataset_type"])  # elimina la columna auxiliar 'dataset_type'
print(f"Columnas después de Transform: {df_transformed.shape[1]} (se quitó 'dataset_type')")  # confirma el nuevo número de columnas

# Load: se guarda el resultado listo para el equipo de ML (esto es justo lo que hace 01_ingestion.py)
processed_dir = Path("../02-phase-1-local-dev-mlops/datasets/processed")  # carpeta de salida para datos procesados
processed_dir.mkdir(parents=True, exist_ok=True)  # crea la carpeta (y las intermedias) si no existe; no falla si ya existe
df_transformed.to_csv(processed_dir / "raw_ingested.csv", index=False)  # guarda el CSV sin agregar la columna de índice de pandas
print(f"Guardado en: {(processed_dir / 'raw_ingested.csv').resolve()}")  # imprime la ruta absoluta del archivo generado

Columnas después de Transform: 24 (se quitó 'dataset_type')
Guardado en: /home/jcarranzaarquitect/Documentos/Machine Learning/3-Mlops/bootcampmlops/02-phase-1-local-dev-mlops/datasets/processed/raw_ingested.csv


## 5. Balance de clases: ¿qué tan frecuente es la rotación?

Una primera pregunta de negocio que cualquier Data Scientist responde antes de modelar: ¿qué proporción de empleados históricamente se fue vs. se quedó?

Cómo se calcula, encadenando cuatro operaciones sobre la columna `Attrition`:

```python
df["Attrition"].value_counts(normalize=True).mul(100).round(2)
#                 └ cuenta cada valor  └ como proporción (0-1)  └ a %  └ 2 decimales
```

**Por qué esto es importante y no un dato curioso:** si el 95% de los empleados se quedara, un modelo que siempre dijera "se queda" tendría 95% de accuracy… y sería completamente inútil. Ese es el **desbalance de clases**, y condiciona tres decisiones que verás más adelante:

| Decisión | Dónde aparece |
|---|---|
| Usar `class_weight="balanced"` al entrenar | Fundamento 3, sección 1 |
| Medir con **recall** en vez de accuracy | Fundamento 3, sección 2 |
| Usar `stratify=y` al partir train/test y en los folds | Fundamento 2 y validación cruzada |

**Qué mirar en la salida:** los dos porcentajes. Cuanto más se alejen de un 50/50, más importantes se vuelven esas tres decisiones.


In [15]:
# value_counts(normalize=True) da la proporción (0-1) de cada valor de "Attrition";
# .mul(100) la convierte a porcentaje y .round(2) la redondea a 2 decimales
attrition_pct = df["Attrition"].value_counts(normalize=True).mul(100).round(2)
print(attrition_pct)  # imprime el % de empleados que se quedaron ("Stayed") vs. se fueron ("Left")

Attrition
Stayed    52.52
Left      47.48
Name: proportion, dtype: float64


## Ideas clave

- El dataset final que usamos en el curso ya pasó por un proceso de ETL y anonimización — en un proyecto real, eso es trabajo previo de Ingeniería de Datos.
- Antes de modelar, conviene entender tamaño, tipos de dato y balance de clases del target.
- `Employee ID` es un identificador pseudonimizado: no es información personal identificable (PII), pero sí una columna a **excluir** como feature (no aporta señal predictiva real).

## Si te perdiste, quédate con esto

Este notebook responde cinco preguntas sobre el dataset, en orden:

```text
1. ¿Puedo leerlo?            → read_csv + head()
2. ¿Cómo son las columnas?   → info()  (¿cuántas son texto?)
3. ¿Cuánto ocupa?            → disco vs. RAM
4. ¿Hay datos personales?    → revisar columnas + ID único
5. ¿Está balanceado?         → % Stayed vs % Left
```

Y deja **un único archivo** para la siguiente etapa: `datasets/processed/raw_ingested.csv`.

| Concepto que aparece aquí | Dónde se vuelve importante |
|---|---|
| Columnas de tipo texto (`object`) | Fundamento 2: hay que convertirlas a números |
| Desbalance de clases | Fundamento 3: `class_weight` y recall |
| `Employee ID` no es una feature | Fundamento 2: se elimina antes de entrenar |
| Tamaño en RAM | Fase 2: cuánta memoria pedirle al pod o al worker |

## Siguiente paso

Continúa con [Fundamento 2: Preparación de Datos](02-preparacion-de-datos.ipynb).
